In [3]:
from pathlib import Path
import sys

# Find the project root from the notebook's working directory.
PROJECT_ROOT = Path.cwd().resolve()

# Run until found the (PROJECT_ROOT / "requirements.txt")
while not (PROJECT_ROOT / "requirements.txt").is_file():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise RuntimeError("Cannot find the project root.")
    PROJECT_ROOT = PROJECT_ROOT.parent

# Allow imports starting with "src".
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"

In [4]:
import numpy as np

train_data = np.load(DATA_DIR / "mnist_train.npz")
test_data = np.load(DATA_DIR / "mnist_test.npz")

X_train, y_train = train_data["X"], train_data["y"]
X_test, y_test = test_data["X"], test_data["y"]

In [5]:
from src.preprocessing.scaler import prepare

from src.preprocessing.image_preprocessing import (
    convert_image,
    clean_images,
    augment_shift
)

In [6]:
print(train_data.files)
print(test_data.files)

['X', 'y']
['X', 'y']


In [7]:
X_train, y_train = train_data['X'], train_data['y']
X_test, y_test = test_data['X'], test_data['y']

In [8]:
X_train, y_train = clean_images(X_train, y_train)

Ảnh trống đã bỏ: 0 | Ảnh trùng đã bỏ: 0 | Còn lại: 60000


1. Scale dữ liệu cho SVM, MLP, Logistic regression 

In [9]:
data = prepare(
    kind="standard",
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test
)

X_train_scaled_standard = data["X_train"]
y_train_standard = data["y_train"]

X_val_scaled_standard = data["X_val"]
y_val_standard = data["y_val"]

X_test_scaled_standard = data["X_test"]
y_test_standard = data["y_test"]

2. Scale dữ liệu cho CNN

In [10]:
data_cnn = prepare(
    kind="cnn",
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test
)

X_train_scaled_cnn = data_cnn["X_train"]
y_train_cnn = data_cnn["y_train"]

X_val_scaled_cnn = data_cnn["X_val"]
y_val_cnn = data_cnn["y_val"]

X_test_scaled_cnn = data_cnn["X_test"]
y_test_cnn = data_cnn["y_test"]

3. Train models

In [11]:
from src.models.logistic_regression import LogisticRegression
from src.models.mlp import MLP
# from src.models.random_forest import RandomForestClassifier
from src.models.cnn import CNN

In [12]:
model_lr = LogisticRegression(
    eta=0.1,
    lam=0.001,
    max_epochs=100,
    batch_size=128,
    seed=42,
    tol=1e-7
)

model_lr.fit(
    X_train_scaled_standard,
    y_train_standard,
    X_val=X_val_scaled_standard,
    y_val=y_val_standard
)

In [13]:
import torch

X_train_t = torch.tensor(
    X_train_scaled_standard, dtype=torch.float32
)
y_train_t = torch.tensor(
    y_train_standard, dtype=torch.long
)

X_val_t = torch.tensor(
    X_val_scaled_standard, dtype=torch.float32
)
y_val_t = torch.tensor(
    y_val_standard, dtype=torch.long
)

In [14]:
model_mlp = MLP(
    sizes=[784,550, 10],
    activation="relu",
    p_drop=0.0,
    
    init="he",
    seed=42
)

model_mlp.fit(
    X_train_t,
    y_train_t,
    epochs=20,
    batch_size=128,
    lr=0.05,
    momentum=0.9,
    weight_decay=0.0,
    validation_data=(X_val_t, y_val_t),
    patience=5,
    seed=42,
    verbose=True
)

Epoch 01/20 | train loss 0.3100 acc 0.9536 | val loss 0.6166 acc 0.9370
Epoch 02/20 | train loss 0.2495 acc 0.9617 | val loss 0.8761 acc 0.9412
Epoch 03/20 | train loss 0.1591 acc 0.9741 | val loss 0.7200 acc 0.9573
Epoch 04/20 | train loss 0.1242 acc 0.9779 | val loss 0.9182 acc 0.9565
Epoch 05/20 | train loss 0.0853 acc 0.9872 | val loss 0.7257 acc 0.9648
Epoch 06/20 | train loss 0.1165 acc 0.9862 | val loss 0.6941 acc 0.9643
Early stopping ở epoch 6 (tốt nhất: epoch 1)


{'train_loss': [0.3099720582917885,
  0.24954104644280892,
  0.15913987559852777,
  0.12420280332918521,
  0.0853330608871248,
  0.11645135269672782],
 'train_acc': [0.9535740740740741,
  0.9617037037037037,
  0.9741111111111111,
  0.9778888888888889,
  0.9872407407407408,
  0.9862222222222222],
 'val_loss': [0.6166377166906992,
  0.876135398944219,
  0.7200030783812205,
  0.9182149569193522,
  0.7257464826107025,
  0.6940745711326599],
 'val_acc': [0.937,
  0.9411666666666667,
  0.9573333333333334,
  0.9565,
  0.9648333333333333,
  0.9643333333333334]}

In [15]:
from sklearn.metrics import accuracy_score

y_pred_lr = model_lr.predict(X_val_scaled_standard)
y_pred_mlp = model_mlp.predict(X_val_t)

acc_lr = accuracy_score(y_val_standard, y_pred_lr)
acc_mlp = accuracy_score(y_val_standard, y_pred_mlp)

print(f"Logistic Regression: {acc_lr:.4f}")
print(f"MLP:                 {acc_mlp:.4f}")

Logistic Regression: 0.9187
MLP:                 0.9370


4. Lưu trọng số

In [21]:
from pathlib import Path
import numpy as np
import joblib

# Tạo thư mục lưu model
WEIGHTS_DIR = Path("src/models/weights")
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

# 1. Lưu Logistic Regression
np.savez(
    WEIGHTS_DIR / "logistic_regression.npz",
    W=model_lr.W_
)

# 2. Lưu MLP NumPy
from pathlib import Path
import torch

WEIGHTS_DIR = Path("src/models/weights")
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

torch.save(
    {
        "sizes": model_mlp.sizes,
        "model_state_dict": model_mlp.model.state_dict(),
    },
    WEIGHTS_DIR / "mlp.pt"
)
import joblib
from pathlib import Path

WEIGHTS = Path("src/models/weights")
WEIGHTS.mkdir(parents=True, exist_ok=True)

scaler_standard = data["tools"]["scaler"]

joblib.dump(
    scaler_standard,
    WEIGHTS / "scaler.pkl"
)

print("Đã lưu scaler.pkl thành công!")


Đã lưu scaler.pkl thành công!


In [23]:
from pathlib import Path

import joblib
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score


from src.preprocessing.scaler import prepare

from pathlib import Path

ROOT = Path.cwd()

TRAIN_PATH = ROOT / "data" / "raw" / "mnist_train.npz"
TEST_PATH = ROOT / "data" / "raw" / "mnist_test.npz"

WEIGHTS = ROOT / "src" / "models" / "weights"
WEIGHTS.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("Train tồn tại:", TRAIN_PATH.exists())
print("Test tồn tại:", TEST_PATH.exists())

TRAIN_PATH = ROOT / "data" / "raw" / "mnist_train.npz"
TEST_PATH = ROOT / "data" / "raw" / "mnist_test.npz"

WEIGHTS = ROOT / "src" / "models" / "weights"
WEIGHTS.mkdir(parents=True, exist_ok=True)


# ==================================================
# 2. Đọc dữ liệu MNIST
# ==================================================
if not TRAIN_PATH.exists():
    raise FileNotFoundError(
        f"Không tìm thấy dữ liệu train: {TRAIN_PATH}"
    )

if not TEST_PATH.exists():
    raise FileNotFoundError(
        f"Không tìm thấy dữ liệu test: {TEST_PATH}"
    )

with np.load(TRAIN_PATH) as train_data:
    X_train = train_data["X"].copy()
    y_train = train_data["y"].copy()

with np.load(TEST_PATH) as test_data:
    X_test = test_data["X"].copy()
    y_test = test_data["y"].copy()

print("=" * 60)
print("THÔNG TIN DỮ LIỆU")
print("=" * 60)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

if len(X_train) != len(y_train):
    raise ValueError(
        "X_train và y_train không có cùng số mẫu."
    )

if len(X_test) != len(y_test):
    raise ValueError(
        "X_test và y_test không có cùng số mẫu."
    )


# ==================================================
# 3. Chia train / validation và chuẩn hóa
# ==================================================
data = prepare(
    kind="standard",
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    val_size=0.1
)

Xtr = data["X_train"]
ytr = data["y_train"]

Xval = data["X_val"]
yval = data["y_val"]

Xte = data["X_test"]
yte = data["y_test"]

# Lấy scaler đã fit trên tập train
scaler = data["tools"]["scaler"]

# Lưu scaler dùng cho hai mô hình sklearn
joblib.dump(
    scaler,
    WEIGHTS / "scaler_sklearn.pkl"
)

print("\nĐã lưu scaler_sklearn.pkl")

print("Train sau chia:", Xtr.shape, ytr.shape)
print("Validation    :", Xval.shape, yval.shape)
print("Test          :", Xte.shape, yte.shape)


# ==================================================
# 4. Hàm đánh giá mô hình
# ==================================================
def evaluate_model(name, model, X_val, y_val, X_test, y_test):
    val_pred = model.predict(X_val)
    test_pred = model.predict(X_test)

    val_acc = accuracy_score(y_val, val_pred)
    test_acc = accuracy_score(y_test, test_pred)

    print(f"\n{name}")
    print("-" * 45)
    print(f"Validation accuracy: {val_acc:.4%}")
    print(f"Test accuracy      : {test_acc:.4%}")

    return {
        "model": name,
        "validation_accuracy": val_acc,
        "test_accuracy": test_acc,
    }


# ==================================================
# 5. Train Logistic Regression của sklearn
# ==================================================
print("\n" + "=" * 60)
print("TRAINING LOGISTIC REGRESSION - SCIKIT-LEARN")
print("=" * 60)

logistic_model = LogisticRegression(
    max_iter=300,
    solver="lbfgs",
    random_state=42
)

logistic_model.fit(Xtr, ytr)

result_lr = evaluate_model(
    name="Logistic Regression (Scikit-learn)",
    model=logistic_model,
    X_val=Xval,
    y_val=yval,
    X_test=Xte,
    y_test=yte
)

# Lưu riêng, không ghi đè model tự cài đặt
joblib.dump(
    logistic_model,
    WEIGHTS / "logistic_regression_sklearn.pkl"
)

print("Đã lưu logistic_regression_sklearn.pkl")


# ==================================================
# 6. Train MLPClassifier của sklearn
# ==================================================
print("\n" + "=" * 60)
print("TRAINING MLP - SCIKIT-LEARN")
print("=" * 60)

mlp_model = MLPClassifier(
    hidden_layer_sizes=(128,),
    activation="relu",
    solver="adam",
    batch_size=128,
    learning_rate_init=0.001,
    max_iter=30,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=5,
    random_state=42,
    verbose=True
)

mlp_model.fit(Xtr, ytr)

result_mlp = evaluate_model(
    name="MLP (Scikit-learn)",
    model=mlp_model,
    X_val=Xval,
    y_val=yval,
    X_test=Xte,
    y_test=yte
)

joblib.dump(
    mlp_model,
    WEIGHTS / "mlp_sklearn.pkl"
)

print("Đã lưu mlp_sklearn.pkl")


# ==================================================
# 7. Tổng kết kết quả
# ==================================================
print("\n" + "=" * 60)
print("SO SÁNH HAI MÔ HÌNH SCIKIT-LEARN")
print("=" * 60)

results = [result_lr, result_mlp]

print(
    f"{'Model':40s}"
    f"{'Validation':>12s}"
    f"{'Test':>12s}"
)

for result in results:
    print(
        f"{result['model']:40s}"
        f"{result['validation_accuracy']:>11.2%}"
        f"{result['test_accuracy']:>12.2%}"
    )

print("\nHoàn tất huấn luyện!")
print(f"Các file đã lưu tại: {WEIGHTS}")


ROOT: /home/nguyenhaianh/Desktop/Classify_Number
Train tồn tại: True
Test tồn tại: True
THÔNG TIN DỮ LIỆU
X_train: (60000, 28, 28)
y_train: (60000,)
X_test : (10000, 28, 28)
y_test : (10000,)

Đã lưu scaler_sklearn.pkl
Train sau chia: (54000, 784) (54000,)
Validation    : (6000, 784) (6000,)
Test          : (10000, 784) (10000,)

TRAINING LOGISTIC REGRESSION - SCIKIT-LEARN

Logistic Regression (Scikit-learn)
---------------------------------------------
Validation accuracy: 91.6000%
Test accuracy      : 91.9800%
Đã lưu logistic_regression_sklearn.pkl

TRAINING MLP - SCIKIT-LEARN
Iteration 1, loss = 0.31918457
Validation score: 0.946111
Iteration 2, loss = 0.12740011
Validation score: 0.960185
Iteration 3, loss = 0.08495593
Validation score: 0.962407
Iteration 4, loss = 0.05881693
Validation score: 0.965185
Iteration 5, loss = 0.04297813
Validation score: 0.967963
Iteration 6, loss = 0.03319010
Validation score: 0.970370
Iteration 7, loss = 0.02540342
Validation score: 0.967963
Iteratio